In [1]:
! pip install torch

In [2]:
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

import torch

In [3]:
X, y = fetch_california_housing(return_X_y=True)

In [4]:
X

array([[   8.3252    ,   41.        ,    6.98412698, ...,    2.55555556,
          37.88      , -122.23      ],
       [   8.3014    ,   21.        ,    6.23813708, ...,    2.10984183,
          37.86      , -122.22      ],
       [   7.2574    ,   52.        ,    8.28813559, ...,    2.80225989,
          37.85      , -122.24      ],
       ...,
       [   1.7       ,   17.        ,    5.20554273, ...,    2.3256351 ,
          39.43      , -121.22      ],
       [   1.8672    ,   18.        ,    5.32951289, ...,    2.12320917,
          39.43      , -121.32      ],
       [   2.3886    ,   16.        ,    5.25471698, ...,    2.61698113,
          39.37      , -121.24      ]], shape=(20640, 8))

The target variable is the median house value for California districts, expressed in hundreds of thousands of dollars ($100,000).

In [5]:
y

array([4.526, 3.585, 3.521, ..., 0.923, 0.847, 0.894], shape=(20640,))

In [6]:
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [7]:
class FCN(torch.nn.Module):
    def __init__(self):
        super().__init__()

        self.fcn = torch.nn.Sequential(
            torch.nn.Linear(8, 20),
            torch.nn.ReLU(),
            torch.nn.Linear(20, 20),
            torch.nn.ReLU(),
            torch.nn.Linear(20, 10),
            torch.nn.ReLU(),
            torch.nn.Linear(10, 1)
        )

    def forward(self, X):
        X = self.fcn(X)
        return X

In [8]:
model = FCN()

In [9]:
X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

In [10]:
train_ds = torch.utils.data.TensorDataset(X_tensor, y_tensor)

In [11]:
mini_batch_size = 64

In [12]:
train_dl = torch.utils.data.DataLoader(train_ds, batch_size=mini_batch_size, shuffle=True, drop_last=False)

In [13]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.003)

In [14]:
sum(p.numel() for p in model.parameters() if p.requires_grad)

821

In [15]:
def fit(epochs, model, optimizer, train_dl):
    loss_func = torch.nn.MSELoss()

    model.train()

    for epoch in range(epochs):  # loop over epochs

        for X_mb, y_mb in train_dl:  # loop over mini-batches

            y_hat = model(X_mb)  # forward pass

            cost = loss_func(y_hat, y_mb)  # compute losses and cost

            cost.backward()  # backward pass (backpropagation)

            optimizer.step()  # update parameters (gradient descent)

            optimizer.zero_grad()

    return model

In [16]:
epochs = 100

In [17]:
fit(epochs, model, optimizer, train_dl)

FCN(
  (fcn): Sequential(
    (0): Linear(in_features=8, out_features=20, bias=True)
    (1): ReLU()
    (2): Linear(in_features=20, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=10, bias=True)
    (5): ReLU()
    (6): Linear(in_features=10, out_features=1, bias=True)
  )
)

In [18]:
with torch.no_grad():
    yhat_train = model(train_ds[:][0])

In [19]:
y_train = train_ds[:][1]

In [20]:
mean_absolute_error(y_train, yhat_train)

0.3974403142929077